# Лабораторная работа №4
## Хеширование

**Выполнил:** Улискин Даниил Евгеньевич  
**Группа:** ФИТ-241

## Задание №1 (5 баллов)

Возьмите реализацию класса `HashTable` из лекционных материалов и выполните следующие доработки:

1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией `len` (переопределите метод `__len__`).
3. Реализуйте работу оператора `in` (переопределите метод `__contains__`).
4. Переопределите метод `put` так, чтобы таблица автоматически увеличивала размер, когда загрузочный фактор становится больше 0,7. Размер должен увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте оператор `del` (метод `__delitem__`) для удаления элемента. Когда загрузочный фактор становится меньше 0,2, размер таблицы должен уменьшаться примерно в два раза до ближайшего подходящего простого числа.

Все выполненные доработки должны быть протестированы.

In [1]:
import math


_EMPTY = object()
_DELETED = object()
_MISSING = object()


def is_prime(number):
    """Проверяет, является ли число простым."""
    if number < 2:
        return False
    if number % 2 == 0:
        return number == 2
    divisor = 3
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 2
    return True


def next_prime(number):
    """Возвращает ближайшее простое число, не меньше заданного."""
    candidate = max(2, math.ceil(number))
    while not is_prime(candidate):
        candidate += 1
    return candidate


def check(description, condition):
    """Печатает результат проверки и останавливает выполнение при ошибке."""
    if not condition:
        raise AssertionError(description)
    print(f'{description:<62} OK')


def check_raises(description, operation, expected_error):
    """Проверяет возникновение ожидаемого исключения."""
    try:
        operation()
    except expected_error:
        print(f'{description:<62} OK ({expected_error.__name__})')
    else:
        raise AssertionError(
            f'{description}: ожидалось исключение {expected_error.__name__}'
        )


class HashTable:
    """Хеш-таблица с открытой адресацией и квадратичным пробированием."""

    MAX_LOAD = 0.7
    MIN_LOAD = 0.2

    def __init__(self, capacity=11):
        capacity = next_prime(max(3, capacity))
        self._min_capacity = capacity
        self._capacity = capacity
        self._slots = [_EMPTY] * capacity
        self._size = 0
        self._used = 0

    def __len__(self):
        return self._size

    def __contains__(self, key):
        _, found = self._locate(key)
        return found

    def __getitem__(self, key):
        index, found = self._locate(key)
        if not found:
            raise KeyError(key)
        return self._slots[index][1]

    def __setitem__(self, key, value):
        self.put(key, value)

    def __delitem__(self, key):
        index, found = self._locate(key)
        if not found:
            raise KeyError(key)

        self._slots[index] = _DELETED
        self._size -= 1

        if (
            self._capacity > self._min_capacity
            and self.load_factor < self.MIN_LOAD
        ):
            minimum_for_items = math.ceil(
                max(1, self._size) / self.MAX_LOAD
            )
            target = next_prime(
                max(
                    self._min_capacity,
                    self._capacity // 2,
                    minimum_for_items,
                )
            )
            if target < self._capacity:
                self._resize(target)
        elif self._used - self._size > self._size:
            # Удаляем накопившиеся маркеры удалённых ячеек.
            self._resize(self._capacity)

    @property
    def capacity(self):
        return self._capacity

    @property
    def load_factor(self):
        return self._size / self._capacity

    def _hash(self, key):
        return hash(key)

    def _probe_indices(self, key):
        """Сначала выдаёт квадратичную последовательность индексов."""
        start = self._hash(key) % self._capacity
        visited = set()

        for step in range(self._capacity):
            index = (start + step * step) % self._capacity
            if index not in visited:
                visited.add(index)
                yield index

        # Для некоторых простых размеров квадраты покрывают не все ячейки.
        # Резервный проход гарантирует корректную работу даже при множестве
        # ключей с одинаковым хешем.
        for index in range(self._capacity):
            if index not in visited:
                yield index

    def _locate(self, key):
        first_deleted = None

        for index in self._probe_indices(key):
            slot = self._slots[index]

            if slot is _EMPTY:
                return (
                    first_deleted if first_deleted is not None else index,
                    False,
                )

            if slot is _DELETED:
                if first_deleted is None:
                    first_deleted = index
            elif slot[0] == key:
                return index, True

        return first_deleted, False

    def put(self, key, value):
        index, found = self._locate(key)

        if found:
            self._slots[index] = (key, value)
            return

        if (self._size + 1) / self._capacity > self.MAX_LOAD:
            self._resize(next_prime(self._capacity * 2))
            index, _ = self._locate(key)

        if index is None:
            self._resize(next_prime(self._capacity * 2))
            index, _ = self._locate(key)

        if self._slots[index] is _EMPTY:
            self._used += 1
        self._slots[index] = (key, value)
        self._size += 1

    def get(self, key, default=_MISSING):
        index, found = self._locate(key)
        if found:
            return self._slots[index][1]
        if default is _MISSING:
            raise KeyError(key)
        return default

    def items(self):
        for slot in self._slots:
            if slot is not _EMPTY and slot is not _DELETED:
                yield slot

    def _insert_without_resize(self, key, value):
        index, found = self._locate(key)
        if index is None:
            raise RuntimeError('Не удалось найти свободную ячейку')
        if self._slots[index] is _EMPTY:
            self._used += 1
        self._slots[index] = (key, value)
        if not found:
            self._size += 1

    def _resize(self, new_capacity):
        old_items = list(self.items())
        self._capacity = next_prime(
            max(self._min_capacity, new_capacity)
        )
        self._slots = [_EMPTY] * self._capacity
        self._size = 0
        self._used = 0
        for key, value in old_items:
            self._insert_without_resize(key, value)


class CollisionKey:
    """Ключ с постоянным хешем для явной проверки коллизий."""

    def __init__(self, value):
        self.value = value

    def __hash__(self):
        return 1

    def __eq__(self, other):
        return (
            isinstance(other, CollisionKey)
            and self.value == other.value
        )


print('ПРОВЕРКА КВАДРАТИЧНОЙ ХЕШ-ТАБЛИЦЫ')

table = HashTable(5)
table['a'] = 10
table['b'] = 20
table['c'] = 30
check('len(table) возвращает число элементов', len(table) == 3)
check("Оператор in находит существующий ключ", 'b' in table)
check("Оператор in отклоняет отсутствующий ключ", 'x' not in table)
check("Получение значения по ключу", table['c'] == 30)

table['b'] = 200
check('Повторная запись обновляет значение', table['b'] == 200)
check('Обновление не увеличивает длину', len(table) == 3)

collision_table = HashTable(11)
collision_keys = [CollisionKey(i) for i in range(6)]
for key in collision_keys:
    collision_table[key] = key.value * 10
check(
    'Квадратичное пробирование разрешает коллизии',
    all(collision_table[key] == key.value * 10 for key in collision_keys),
)

growth_table = HashTable(5)
start_capacity = growth_table.capacity
for number in range(12):
    growth_table[number] = number ** 2
grown_capacity = growth_table.capacity
check('При загрузке > 0,7 таблица увеличилась', grown_capacity > start_capacity)
check('Новый размер является простым числом', is_prime(grown_capacity))
check(
    'После увеличения все данные сохранены',
    all(growth_table[number] == number ** 2 for number in range(12)),
)

for number in range(10):
    del growth_table[number]
check('Оператор del удаляет элементы', 0 not in growth_table)
check('При загрузке < 0,2 таблица уменьшилась', growth_table.capacity < grown_capacity)
check(
    'Оставшиеся элементы сохранены после уменьшения',
    growth_table[10] == 100 and growth_table[11] == 121,
)
check_raises(
    'Удаление отсутствующего ключа вызывает ошибку',
    lambda: growth_table.__delitem__(999),
    KeyError,
)

print(
    f'Итог: длина = {len(growth_table)}, '
    f'ёмкость = {growth_table.capacity}, '
    f'загрузка = {growth_table.load_factor:.2f}'
)

ПРОВЕРКА КВАДРАТИЧНОЙ ХЕШ-ТАБЛИЦЫ
len(table) возвращает число элементов                          OK
Оператор in находит существующий ключ                          OK
Оператор in отклоняет отсутствующий ключ                       OK
Получение значения по ключу                                    OK
Повторная запись обновляет значение                            OK
Обновление не увеличивает длину                                OK
Квадратичное пробирование разрешает коллизии                   OK
При загрузке > 0,7 таблица увеличилась                         OK
Новый размер является простым числом                           OK
После увеличения все данные сохранены                          OK
Оператор del удаляет элементы                                  OK
При загрузке < 0,2 таблица уменьшилась                         OK
Оставшиеся элементы сохранены после уменьшения                 OK
Удаление отсутствующего ключа вызывает ошибку                  OK (KeyError)
Итог: длина = 2, ёмкость = 5, з

## Задание №2 (5 баллов)

Возьмите реализацию класса `HashTable` из лекционных материалов и выполните следующие доработки:

1. Переделайте разрешение коллизий с открытой адресации на метод цепочек. В каждом слоте храните связный список, реализованный классом `UnorderedList` из лабораторной работы №3.
2. Реализуйте `len(table)`.
3. Реализуйте оператор `key in table`.
4. Автоматически увеличивайте таблицу примерно в два раза до ближайшего подходящего простого числа, если загрузочный фактор становится больше 0,7.
5. Реализуйте оператор `del table[key]`. Автоматически уменьшайте таблицу примерно в два раза до ближайшего подходящего простого числа, если загрузочный фактор становится меньше 0,2.

Все выполненные доработки должны быть протестированы.

In [2]:
class Node:
    """Узел односвязного списка из лабораторной работы №3."""

    def __init__(self, data, next_node=None):
        self.data = data
        self.next = next_node


class UnorderedList:
    """Односвязный список для хранения цепочки коллизий."""

    def __init__(self):
        self.head = None
        self._size = 0

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.data
            current = current.next

    def __len__(self):
        return self._size

    def add(self, item):
        self.head = Node(item, self.head)
        self._size += 1

    def remove(self, item):
        previous = None
        current = self.head

        while current is not None and current.data is not item:
            previous, current = current, current.next

        if current is None:
            raise ValueError('Элемент отсутствует в списке')

        if previous is None:
            self.head = current.next
        else:
            previous.next = current.next
        self._size -= 1


class ChainedHashTable:
    """Хеш-таблица, разрешающая коллизии методом цепочек."""

    MAX_LOAD = 0.7
    MIN_LOAD = 0.2

    def __init__(self, capacity=11):
        capacity = next_prime(max(3, capacity))
        self._min_capacity = capacity
        self._capacity = capacity
        self._buckets = [UnorderedList() for _ in range(capacity)]
        self._size = 0

    def __len__(self):
        return self._size

    def __contains__(self, key):
        return self._find_entry(key) is not None

    def __getitem__(self, key):
        entry = self._find_entry(key)
        if entry is None:
            raise KeyError(key)
        return entry[1]

    def __setitem__(self, key, value):
        self.put(key, value)

    def __delitem__(self, key):
        bucket = self._buckets[self._bucket_index(key)]
        entry = self._find_entry(key)

        if entry is None:
            raise KeyError(key)

        bucket.remove(entry)
        self._size -= 1

        if (
            self._capacity > self._min_capacity
            and self.load_factor < self.MIN_LOAD
        ):
            minimum_for_items = math.ceil(
                max(1, self._size) / self.MAX_LOAD
            )
            target = next_prime(
                max(
                    self._min_capacity,
                    self._capacity // 2,
                    minimum_for_items,
                )
            )
            if target < self._capacity:
                self._resize(target)

    @property
    def capacity(self):
        return self._capacity

    @property
    def load_factor(self):
        return self._size / self._capacity

    def _hash(self, key):
        return hash(key)

    def _bucket_index(self, key):
        return self._hash(key) % self._capacity

    def _find_entry(self, key):
        bucket = self._buckets[self._bucket_index(key)]
        for entry in bucket:
            if entry[0] == key:
                return entry
        return None

    def put(self, key, value):
        entry = self._find_entry(key)
        if entry is not None:
            entry[1] = value
            return

        if (self._size + 1) / self._capacity > self.MAX_LOAD:
            self._resize(next_prime(self._capacity * 2))

        self._buckets[self._bucket_index(key)].add([key, value])
        self._size += 1

    def get(self, key, default=_MISSING):
        entry = self._find_entry(key)
        if entry is not None:
            return entry[1]
        if default is _MISSING:
            raise KeyError(key)
        return default

    def items(self):
        for bucket in self._buckets:
            for key, value in bucket:
                yield key, value

    def _resize(self, new_capacity):
        old_items = list(self.items())
        self._capacity = next_prime(
            max(self._min_capacity, new_capacity)
        )
        self._buckets = [
            UnorderedList() for _ in range(self._capacity)
        ]
        self._size = 0
        for key, value in old_items:
            self._buckets[self._bucket_index(key)].add([key, value])
            self._size += 1


print('ПРОВЕРКА ХЕШ-ТАБЛИЦЫ С ЦЕПОЧКАМИ')

chain_table = ChainedHashTable(5)
chain_keys = [CollisionKey(i) for i in range(8)]
for key in chain_keys:
    chain_table[key] = f'value-{key.value}'

check('Все ключи с одинаковым хешем сохранены', len(chain_table) == 8)
check(
    'Все элементы цепочки доступны',
    all(
        chain_table[key] == f'value-{key.value}'
        for key in chain_keys
    ),
)
check('Оператор in работает с цепочками', chain_keys[4] in chain_table)

chain_table[chain_keys[4]] = 'updated'
check('Значение внутри цепочки обновляется', chain_table[chain_keys[4]] == 'updated')
check('Обновление не меняет len', len(chain_table) == 8)

large_capacity = chain_table.capacity
for key in chain_keys[:7]:
    del chain_table[key]
check('del удаляет элементы из цепочек', len(chain_table) == 1)
check('Таблица уменьшилась при загрузке < 0,2', chain_table.capacity < large_capacity)
check('Оставшийся ключ сохранился', chain_keys[7] in chain_table)
check_raises(
    'Удаление отсутствующего ключа вызывает ошибку',
    lambda: chain_table.__delitem__('нет'),
    KeyError,
)

print(
    f'Итог: длина = {len(chain_table)}, '
    f'ёмкость = {chain_table.capacity}, '
    f'длина оставшейся цепочки = '
    f'{len(chain_table._buckets[chain_table._bucket_index(chain_keys[7])])}'
)

ПРОВЕРКА ХЕШ-ТАБЛИЦЫ С ЦЕПОЧКАМИ
Все ключи с одинаковым хешем сохранены                         OK
Все элементы цепочки доступны                                  OK
Оператор in работает с цепочками                               OK
Значение внутри цепочки обновляется                            OK
Обновление не меняет len                                       OK
del удаляет элементы из цепочек                                OK
Таблица уменьшилась при загрузке < 0,2                         OK
Оставшийся ключ сохранился                                     OK
Удаление отсутствующего ключа вызывает ошибку                  OK (KeyError)
Итог: длина = 1, ёмкость = 5, длина оставшейся цепочки = 1


## Задание №3 (2 балла)

Переделайте класс `HashTable`, чтобы в качестве ключей можно было использовать строки.

In [3]:
class StringHashTable(HashTable):
    """Хеш-таблица со строковыми ключами и детерминированным хешем."""

    def _hash(self, key):
        if not isinstance(key, str):
            raise TypeError('Ключ должен быть строкой')

        value = 0
        for character in key:
            value = value * 257 + ord(character)
        return value


print('ПРОВЕРКА СТРОКОВЫХ КЛЮЧЕЙ')

string_table = StringHashTable(5)
string_table['имя'] = 'Даниил'
string_table['группа'] = 'ФИТ-241'
string_table['language'] = 'Python'

check('Русский строковый ключ сохраняется', string_table['имя'] == 'Даниил')
check('Английский строковый ключ сохраняется', string_table['language'] == 'Python')
check('Оператор in работает со строками', 'группа' in string_table)

string_table['группа'] = 'ФИТ-241 (обновлено)'
check(
    'Повторный строковый ключ обновляет значение',
    string_table['группа'] == 'ФИТ-241 (обновлено)',
)

del string_table['language']
check('Строковый ключ удаляется', 'language' not in string_table)
check_raises(
    'Нестроковый ключ отклоняется',
    lambda: string_table.put(123, 'ошибка'),
    TypeError,
)

print('Содержимое:', sorted(string_table.items()))

ПРОВЕРКА СТРОКОВЫХ КЛЮЧЕЙ
Русский строковый ключ сохраняется                             OK
Английский строковый ключ сохраняется                          OK
Оператор in работает со строками                               OK
Повторный строковый ключ обновляет значение                    OK
Строковый ключ удаляется                                       OK
Нестроковый ключ отклоняется                                   OK (TypeError)
Содержимое: [('группа', 'ФИТ-241 (обновлено)'), ('имя', 'Даниил')]


## Задание №4 (2 балла)

Дана строка русского текста, состоящая из слов и пробелов. Словом считается последовательность русских букв; слова разделены одним или большим числом пробелов.

Для каждого слова определите порядковый номер его вхождения в текст именно в той форме, в которой указано слово. Для первого вхождения выведите `1`, для второго вхождения того же слова - `2` и так далее.

Для решения используйте класс `HashTable` из задания №3.

**Пример**

Ввод:

`Раз раз раз как меня слышно Повторяю раз раз раз Повторяю`

Вывод:

`1 1 2 1 1 1 1 3 4 5 2`

In [4]:
def word_occurrence_numbers(text):
    """Возвращает номера вхождений слов с учётом регистра."""
    counters = StringHashTable()
    result = []

    for word in text.split():
        current_number = counters.get(word, 0) + 1
        counters[word] = current_number
        result.append(current_number)

    return result


sample_text = (
    'Раз раз раз как меня слышно '
    'Повторяю раз раз раз Повторяю'
)
sample_result = word_occurrence_numbers(sample_text)
expected_result = [1, 1, 2, 1, 1, 1, 1, 3, 4, 5, 2]

print('Ввод: ', sample_text)
print('Вывод:', *sample_result)

check('Результат совпадает с примером', sample_result == expected_result)
check(
    'Регистр учитывается',
    word_occurrence_numbers('мир мир Мир мир') == [1, 2, 1, 3],
)
check(
    'Несколько пробелов корректно разделяют слова',
    word_occurrence_numbers('кот   кот  пёс') == [1, 2, 1],
)

Ввод:  Раз раз раз как меня слышно Повторяю раз раз раз Повторяю
Вывод: 1 1 2 1 1 1 1 3 4 5 2
Результат совпадает с примером                                 OK
Регистр учитывается                                            OK
Несколько пробелов корректно разделяют слова                   OK


## Задание №5 (3 балла)

Напишите программу, имитирующую регистрацию и авторизацию. Для каждого пользователя программа должна сохранять логин, хеш пароля и «соль». Данные можно хранить в БД или файле.

### Сохранение пароля

1. Сгенерируйте криптографически стойкую случайную соль при помощи модуля `secrets`. Для SHA-256/scrypt используется не менее 32 случайных байт.
2. Добавьте соль к паролю и вычислите хеш с помощью функции `scrypt` из модуля `hashlib`.
3. Сохраните логин, соль и хеш в файл.

### Проверка пароля

1. Извлеките соль и хеш пользователя из файла.
2. Вычислите хеш введённого пароля с той же солью и теми же параметрами.
3. Безопасно сравните полученный хеш с сохранённым. Совпадение означает правильный пароль.

In [5]:
import base64
import hashlib
import hmac
import json
import secrets
import tempfile
from pathlib import Path


class UserStore:
    """Файловое хранилище логинов, солей и хешей паролей."""

    SALT_SIZE = 32
    HASH_SIZE = 32
    SCRYPT_N = 2 ** 14
    SCRYPT_R = 8
    SCRYPT_P = 1

    def __init__(self, file_path):
        self.file_path = Path(file_path)

    def _load(self):
        if not self.file_path.exists():
            return {}
        return json.loads(self.file_path.read_text(encoding='utf-8'))

    def _save(self, users):
        self.file_path.parent.mkdir(parents=True, exist_ok=True)
        self.file_path.write_text(
            json.dumps(users, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )

    @classmethod
    def _derive_hash(cls, password, salt):
        return hashlib.scrypt(
            password.encode('utf-8'),
            salt=salt,
            n=cls.SCRYPT_N,
            r=cls.SCRYPT_R,
            p=cls.SCRYPT_P,
            dklen=cls.HASH_SIZE,
        )

    def register(self, login, password):
        if not login or not password:
            raise ValueError('Логин и пароль не должны быть пустыми')

        users = self._load()
        if login in users:
            raise ValueError('Пользователь уже зарегистрирован')

        salt = secrets.token_bytes(self.SALT_SIZE)
        password_hash = self._derive_hash(password, salt)
        users[login] = {
            'salt': base64.b64encode(salt).decode('ascii'),
            'hash': base64.b64encode(password_hash).decode('ascii'),
        }
        self._save(users)

    def authenticate(self, login, password):
        users = self._load()
        record = users.get(login)
        if record is None:
            return False

        salt = base64.b64decode(record['salt'])
        expected_hash = base64.b64decode(record['hash'])
        actual_hash = self._derive_hash(password, salt)
        return hmac.compare_digest(actual_hash, expected_hash)


print('ПРОВЕРКА РЕГИСТРАЦИИ И АВТОРИЗАЦИИ')

with tempfile.TemporaryDirectory() as temporary_directory:
    database_path = Path(temporary_directory) / 'users.json'
    users = UserStore(database_path)

    users.register('daniel', 'Strong password 2026!')

    check(
        'Правильный пароль разрешает вход',
        users.authenticate('daniel', 'Strong password 2026!'),
    )
    check(
        'Неправильный пароль отклоняется',
        not users.authenticate('daniel', 'wrong password'),
    )
    check(
        'Неизвестный пользователь отклоняется',
        not users.authenticate('unknown', 'password'),
    )

    stored_data = json.loads(database_path.read_text(encoding='utf-8'))
    stored_text = database_path.read_text(encoding='utf-8')
    record = stored_data['daniel']

    check(
        'Открытый пароль не записан в файл',
        'Strong password 2026!' not in stored_text,
    )
    check(
        'Соль содержит 32 случайных байта',
        len(base64.b64decode(record['salt'])) == 32,
    )
    check(
        'Хеш содержит 32 байта',
        len(base64.b64decode(record['hash'])) == 32,
    )
    check_raises(
        'Повторная регистрация того же логина запрещена',
        lambda: users.register('daniel', 'another password'),
        ValueError,
    )

print(
    'В файл сохраняются только логин, соль и хеш; '
    'открытый пароль не сохраняется.'
)

ПРОВЕРКА РЕГИСТРАЦИИ И АВТОРИЗАЦИИ


Правильный пароль разрешает вход                               OK
Неправильный пароль отклоняется                                OK
Неизвестный пользователь отклоняется                           OK
Открытый пароль не записан в файл                              OK
Соль содержит 32 случайных байта                               OK
Хеш содержит 32 байта                                          OK
Повторная регистрация того же логина запрещена                 OK (ValueError)
В файл сохраняются только логин, соль и хеш; открытый пароль не сохраняется.


## Задание №6 (3 балла)

Напишите программу, которая принимает от пользователя путь к директории. Для всех файлов из этой директории должен быть вычислен хеш. Программа должна найти и вывести все дубликаты - файлы с одинаковым хешем.

In [6]:
def sha256_file(file_path, chunk_size=64 * 1024):
    """Вычисляет SHA-256 файла порциями, не загружая файл целиком."""
    digest = hashlib.sha256()
    with Path(file_path).open('rb') as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def find_duplicate_files(directory):
    """Возвращает группы файлов с одинаковым содержимым."""
    root = Path(directory)
    if not root.is_dir():
        raise NotADirectoryError(root)

    files_by_hash = {}
    for file_path in sorted(root.iterdir(), key=lambda path: path.name):
        if file_path.is_file():
            digest = sha256_file(file_path)
            files_by_hash.setdefault(digest, []).append(file_path)

    duplicate_groups = [
        paths
        for paths in files_by_hash.values()
        if len(paths) > 1
    ]
    return sorted(
        duplicate_groups,
        key=lambda paths: paths[0].name,
    )


def print_duplicate_files(directory):
    duplicate_groups = find_duplicate_files(directory)

    if not duplicate_groups:
        print('Дубликаты не найдены.')
        return

    for group_number, paths in enumerate(duplicate_groups, start=1):
        print(f'Группа дубликатов №{group_number}:')
        for path in paths:
            print(f'  {path.name}')


print('ПРОВЕРКА ПОИСКА ФАЙЛОВ-ДУБЛИКАТОВ')

with tempfile.TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)

    (root / 'report.txt').write_text(
        'Одинаковое содержимое',
        encoding='utf-8',
    )
    (root / 'report-copy.txt').write_text(
        'Одинаковое содержимое',
        encoding='utf-8',
    )
    (root / 'unique.txt').write_text(
        'Уникальный файл',
        encoding='utf-8',
    )
    (root / 'empty-a.txt').write_bytes(b'')
    (root / 'empty-b.txt').write_bytes(b'')

    groups = find_duplicate_files(root)
    name_groups = {
        frozenset(path.name for path in group)
        for group in groups
    }

    check('Найдены ровно две группы дубликатов', len(groups) == 2)
    check(
        'Найдены два одинаковых текстовых файла',
        frozenset({'report.txt', 'report-copy.txt'}) in name_groups,
    )
    check(
        'Найдены два одинаковых пустых файла',
        frozenset({'empty-a.txt', 'empty-b.txt'}) in name_groups,
    )
    check(
        'Уникальный файл не отмечен как дубликат',
        all(
            'unique.txt' not in {path.name for path in group}
            for group in groups
        ),
    )

    print()
    print_duplicate_files(root)

# Для ручной проверки директории:
# directory = input('Введите путь к директории: ')
# print_duplicate_files(directory)

ПРОВЕРКА ПОИСКА ФАЙЛОВ-ДУБЛИКАТОВ
Найдены ровно две группы дубликатов                            OK
Найдены два одинаковых текстовых файла                         OK
Найдены два одинаковых пустых файла                            OK
Уникальный файл не отмечен как дубликат                        OK

Группа дубликатов №1:
  empty-a.txt
  empty-b.txt
Группа дубликатов №2:
  report-copy.txt
  report.txt
